# NB 1 — LLM vs. LLM + a tool
**Goal:** see *why tools, not a cleverer prompt, carry much of the value.*

We ask a small dosing arithmetic question two ways: the model alone, and the model + a calculator tool.
The model alone may get it right or wrong — and it won't tell you which. The tool makes the answer deterministic and checkable. (Runs in MOCK mode with no API key; set a key/model to make it real.)

In [1]:

import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model: pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages):
    """Scripted: the 'model alone' skips the lb->kg conversion (a realistic
    silent error); with a real model it may or may not — and it won't tell you
    which. The point is that the TOOL makes the answer deterministic & checkable."""
    last = messages[-1]["content"].lower()
    if "just the number" in last and "calculator" not in last:
        return "176"   # wrong: forgot to convert pounds to kilograms
    return "I need a calculator for that."


Backend: REAL model = openai/gpt-4o-mini


### 1) The model alone
A weight-based dose: enoxaparin **1 mg/kg** for a patient weighing **176 lb**. Correct answer requires converting lb → kg (176 / 2.2046 = 79.8 kg → **79.8 mg**).

In [2]:
q = ("A patient weighs 176 lb. Enoxaparin dose is 1 mg/kg. "
     "What is the dose in mg? Answer with just the number.")
ans = chat([{"role":"user","content":q}])
print("Model alone ->", ans, "mg")
print("Correct      -> 79.8 mg  (176 / 2.2046)")

Model alone -> 80 mg
Correct      -> 79.8 mg  (176 / 2.2046)


### 2) The model + a calculator tool
Give the model one tool. Now the arithmetic is offloaded to code that cannot hallucinate.

In [3]:
def calculator(expr: str) -> float:
    """A safe arithmetic tool."""
    import ast, operator as op
    ops={ast.Add:op.add,ast.Sub:op.sub,ast.Mult:op.mul,ast.Div:op.truediv,ast.USub:op.neg,ast.Pow:op.pow}
    def ev(n):
        if isinstance(n,ast.Num): return n.n
        if isinstance(n,ast.BinOp): return ops[type(n.op)](ev(n.left),ev(n.right))
        if isinstance(n,ast.UnaryOp): return ops[type(n.op)](ev(n.operand))
        raise ValueError("unsupported")
    return ev(ast.parse(expr,mode="eval").body)

# The "agent" step: decide the computation, then let the tool do it exactly.
dose = calculator("176 / 2.2046 * 1")
print(f"Model + tool -> {dose:.1f} mg   (grounded, reproducible, checkable)")

Model + tool -> 79.8 mg   (grounded, reproducible, checkable)


/var/folders/jp/llt8_8113vj5lh6ddtzp1s100000gn/T/ipykernel_56883/2841411800.py:6: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(n,ast.Num): return n.n
/var/folders/jp/llt8_8113vj5lh6ddtzp1s100000gn/T/ipykernel_56883/2841411800.py:6: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  if isinstance(n,ast.Num): return n.n


### Takeaway
The tool didn't make the model *smarter* — it made the result **reliable**. In a real workflow, the reliability of tools, retrieval/grounding, and verification usually matters more than which reasoning-style prompt you picked. This is the foundation the next two notebooks build on.

*Try:* set `OPENAI_API_KEY` (or point `OPENAI_BASE_URL` at a local open-weight model) and re-run — watch whether the model-alone answer changes run to run.